# Notebook 3: Closed-Set and OOD Confidence Evaluation

This notebook evaluates the certainty of the trained baseline models on both in-distribution test data and out-of-distribution (OOD) inputs. The in-distribution test set contains examples from the known classes, while the OOD datasets contain examples outside the classes used to train the models. This comparison shows whether the models assign lower confidence to unfamiliar inputs or remain overconfident when making predictions.

The workflow loads the saved model checkpoints, prepares the evaluation images using the same preprocessing pipeline as training, and runs inference on the in-distribution and OOD datasets. For every image, the model produces a prediction among the known classes and a confidence score based on its output probabilities. The notebook compares these confidence scores, including maximum softmax probabilities, across the in-distribution and OOD groups.

No abstention or rejection rule is applied in this notebook. OOD examples are therefore forced into one of the known classes, allowing the experiment to measure the models' raw certainty and expose overconfident OOD predictions. The resulting logits, predictions, and confidence summaries provide the baseline for the later notebooks, where abstention methods can be introduced and evaluated.

## Import Libraries

In [1]:
from pathlib import Path
import json
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (
    mobilenet_v3_small,
    efficientnet_b0,
)
from PIL import Image
import numpy as np

In [2]:
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Device:", device)

Device: mps


## Load Test ID dataset and OOD dataset Manifests

In [3]:
SPLIT_DIR = Path("./splits")

test_id = pd.read_csv(SPLIT_DIR / "test_id.csv")
ood_grape = pd.read_csv(SPLIT_DIR / "ood_grape.csv")
ood_corn = pd.read_csv(SPLIT_DIR / "ood_corn.csv")
ood_nonplant = pd.read_csv(SPLIT_DIR / "ood_nonplant.csv")

with open(SPLIT_DIR / "class_to_idx.json", "r") as f:
    class_to_idx = json.load(f)

idx_to_class = {
    idx: class_name
    for class_name, idx in class_to_idx.items()
}

NUM_CLASSES = len(class_to_idx)

print("Classes:", NUM_CLASSES)
print("ID test:", len(test_id))
print("Grape OOD:", len(ood_grape))
print("Corn OOD:", len(ood_corn))
print("Non-plant OOD:", len(ood_nonplant))

Classes: 30
ID test: 9159
Grape OOD: 4062
Corn OOD: 3852
Non-plant OOD: 600


## Create Dataset

In [4]:
class InferenceDataset(Dataset):
    def __init__(self, dataframe, transform=None, has_label=True):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.has_label = has_label

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["path"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        if self.has_label:
            label = class_to_idx[row["class_name"]]
            return image, label, row["path"]

        return image, row["path"]

In [5]:
# Transformation for pre-trained model's expected input format
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [ ]:
# Create dataset objects for test, grape OOD, corn OOD and non-plant OOD
test_dataset = InferenceDataset(
    test_id,
    transform=eval_transform,
    has_label=True,
)

grape_dataset = InferenceDataset(
    ood_grape,
    transform=eval_transform,
    has_label=False,
)

corn_dataset = InferenceDataset(
    ood_corn,
    transform=eval_transform,
    has_label=False,
)

nonplant_dataset = InferenceDataset(
    ood_nonplant,
    transform=eval_transform,
    has_label=False,
)

In [ ]:
# Create data loaders objects for test, grape OOD, corn OOD and non-plant OOD

BATCH_SIZE = 64

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

grape_loader = DataLoader(
    grape_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

corn_loader = DataLoader(
    corn_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

nonplant_loader = DataLoader(
    nonplant_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

## Recreate Model Architectures

In [ ]:
# General model creation function
def create_model(model_name, num_classes):
    if model_name == "mobilenet_v3_small":
        model = mobilenet_v3_small(weights=None)
        model.classifier[3] = nn.Linear(
            model.classifier[3].in_features,
            num_classes,
        )

    elif model_name == "efficientnet_b0":
        model = efficientnet_b0(weights=None)
        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features,
            num_classes,
        )

    else:
        raise ValueError(model_name)

    return model

In [9]:
MODEL_DIR = Path("./models")

## Generic Inference Function

In [10]:
@torch.no_grad()
def infer_id(model, loader, device):
    all_logits = []
    all_labels = []
    all_paths = []

    model.eval()

    for images, labels, paths in loader:
        images = images.to(device)

        logits = model(images)

        all_logits.append(logits.cpu())
        all_labels.extend(labels.numpy())
        all_paths.extend(paths)

    return {
        "logits": torch.cat(all_logits),
        "labels": torch.tensor(all_labels),
        "paths": all_paths,
    }

@torch.no_grad()
def infer_ood(model, loader, device):
    all_logits = []
    all_paths = []

    model.eval()

    for images, paths in loader:
        images = images.to(device)

        logits = model(images)

        all_logits.append(logits.cpu())
        all_paths.extend(paths)

    return {
        "logits": torch.cat(all_logits),
        "paths": all_paths,
    }

## Seeded closed-set and OOD evaluation

The saved baseline checkpoints are evaluated independently for seeds 14, 27, and 42. Each seed produces separate logits, summaries, and metadata-compatible artifacts for the post-hoc notebooks.

In [ ]:
SEEDS = [14, 27, 42]
LOGIT_DIR = Path("./logits")
RESULT_DIR = Path("./results")
LOGIT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# Used to load the models with the trained weights from notebook 02
def load_model(model_name, seed):
    model = create_model(model_name, NUM_CLASSES)
    checkpoint = MODEL_DIR / f"{model_name}_seed{seed}.pt"
    model.load_state_dict(torch.load(checkpoint, map_location="cpu"))
    return model.to(device).eval()

def save_inference_outputs(prefix, outputs, seed):
    torch.save(outputs, LOGIT_DIR / f"{prefix}_seed{seed}.pt")

seed_summaries = []
for seed in SEEDS:
    mobilenet = load_model("mobilenet_v3_small", seed)
    efficientnet = load_model("efficientnet_b0", seed)

    # Get raw model outputs (logits)
    model_outputs = {
        "mobilenet": {
            "id": infer_id(mobilenet, test_loader, device),
            "grape": infer_ood(mobilenet, grape_loader, device),
            "corn": infer_ood(mobilenet, corn_loader, device),
            "nonplant": infer_ood(mobilenet, nonplant_loader, device),
        },
        "efficientnet": {
            "id": infer_id(efficientnet, test_loader, device),
            "grape": infer_ood(efficientnet, grape_loader, device),
            "corn": infer_ood(efficientnet, corn_loader, device),
            "nonplant": infer_ood(efficientnet, nonplant_loader, device),
        },
    }

    # Use softmax to convert logits into probilities. Then get the max probability (class) for each prediction
    for model_prefix, outputs in model_outputs.items():
        for dataset_name, output in outputs.items():
            save_inference_outputs(f"{model_prefix}_{dataset_name}", output, seed)

        id_confidence = torch.softmax(outputs["id"]["logits"], dim=1).max(dim=1).values
        ood_confidences = {
            name: torch.softmax(outputs[name]["logits"], dim=1).max(dim=1).values
            for name in ("grape", "corn", "nonplant")
        }
        for dataset_name, confidence in [("ID", id_confidence), *[(f"{name.title()} OOD", value) for name, value in ood_confidences.items()]]:
            seed_summaries.append({
                "model": "MobileNetV3-Small" if model_prefix == "mobilenet" else "EfficientNet-B0",
                "seed": seed,
                "dataset": dataset_name,
                "mean_max_softmax": float(confidence.mean()),
            })

closed_set_summary = pd.DataFrame(seed_summaries)
closed_set_summary.to_csv(RESULT_DIR / "closed_set_ood_summary_seeded.csv", index=False)
closed_set_summary

,model,seed,dataset,mean_max_softmax
0,MobileNetV3-Small,14,ID,0.992303
1,MobileNetV3-Small,14,Grape OOD,0.764074
2,MobileNetV3-Small,14,Corn OOD,0.695729
3,MobileNetV3-Small,14,Nonplant OOD,0.708113
4,EfficientNet-B0,14,ID,0.996520
5,EfficientNet-B0,14,Grape OOD,0.754334
6,EfficientNet-B0,14,Corn OOD,0.609079
7,EfficientNet-B0,14,Nonplant OOD,0.728996
8,MobileNetV3-Small,27,ID,0.994051
9,MobileNetV3-Small,27,Grape OOD,0.748458
